# Jupyter como cliente de Apache Airflow

En este notebook demuestro cómo usar Jupyter como cliente de Airflow para:
1. **Lanzar DAGs** mediante la API REST
2. **Consultar ejecuciones**, estados y logs mediante la API REST
3. **Ejecutar comandos CLI** de Airflow directamente desde Jupyter
4. **Explicar las ventajas** de usar Jupyter como cliente de Airflow

## 1. Configuración inicial

Primero configuro la conexión con la API REST de Airflow.

In [1]:
import requests
import json
from datetime import datetime

# Configuración de la API REST de Airflow
# Desde el contenedor Jupyter, Airflow webserver está accesible por nombre de servicio
AIRFLOW_API_URL = "http://airflow-webserver:8080/api/v1"
AIRFLOW_USER = "admin"
AIRFLOW_PASS = "admin"

# Sesión con autenticación básica
session = requests.Session()
session.auth = (AIRFLOW_USER, AIRFLOW_PASS)
session.headers.update({'Content-Type': 'application/json'})

print(" Configuración de conexión a Airflow completada")
print(f"   URL: {AIRFLOW_API_URL}")
print(f"   Usuario: {AIRFLOW_USER}")

 Configuración de conexión a Airflow completada
   URL: http://airflow-webserver:8080/api/v1
   Usuario: admin


## 2. Listar DAGs disponibles

Antes de lanzar un DAG, voy a comprobar cuáles están disponibles en mi entorno Airflow.

In [2]:
# Listar todos los DAGs disponibles
response = session.get(f"{AIRFLOW_API_URL}/dags")

if response.status_code == 200:
    dags = response.json()['dags']
    print(f" DAGs disponibles ({len(dags)}):")
    print("-" * 60)
    for dag in dags:
        estado = '▶️ Activo' if not dag['is_paused'] else '⏸️ Pausado'
        print(f"  • {dag['dag_id']:30s} | {estado}")
    print("-" * 60)
else:
    print(f" Error al listar DAGs: {response.status_code}")
    print(response.text)

 DAGs disponibles (3):
------------------------------------------------------------
  • dag_ejemplo                    | ▶️ Activo
  • dag_papermill                  | ▶️ Activo
  • workflow_video                 | ▶️ Activo
------------------------------------------------------------


## 3. Lanzar un DAG mediante la API REST

Lanzo el DAG`dag_ejemplo` usando un`POST` a la API REST de Airflow, Es lo mismo que hacer clic en "Trigger DAG" en la interfaz web.

In [3]:
# Lanzar el DAG 'dag_ejemplo' mediante la API REST
dag_id = "dag_ejemplo"

# Cuerpo de la petición - podemos pasar configuración JSON
payload = {
    "conf": {},  # Configuración opcional para el DAG
    "note": "Lanzado desde Jupyter Notebook"
}

response = session.post(
    f"{AIRFLOW_API_URL}/dags/{dag_id}/dagRuns",
    json=payload
)

if response.status_code in [200, 201]:
    dag_run = response.json()
    print(f" DAG '{dag_id}' lanzado correctamente!")
    print(f"   DAG Run ID: {dag_run['dag_run_id']}")
    print(f"   Estado: {dag_run['state']}")
    print(f"   Fecha inicio: {dag_run['execution_date']}")
    
    # Guardamos el dag_run_id para consultas posteriores
    dag_run_id = dag_run['dag_run_id']
else:
    print(f" Error al lanzar DAG: {response.status_code}")
    print(response.text)

 DAG 'dag_ejemplo' lanzado correctamente!
   DAG Run ID: manual__2026-03-21T21:29:44.824365+00:00
   Estado: queued
   Fecha inicio: 2026-03-21T21:29:44.824365+00:00


## 4. Consultar ejecuciones del DAG

Consulto todas las ejecuciones del DAG y su estado actual.

In [4]:
import time

# Esperamos unos segundos para que el DAG se ejecute
print("⏳ Esperando 10 segundos para que el DAG se ejecute...")
time.sleep(10)

# Consultar todas las ejecuciones del DAG
response = session.get(f"{AIRFLOW_API_URL}/dags/{dag_id}/dagRuns")

if response.status_code == 200:
    dag_runs = response.json()['dag_runs']
    print(f" Ejecuciones del DAG '{dag_id}' ({len(dag_runs)} total):")
    print("-" * 80)
    for run in dag_runs:
        estado_emoji = {'success': '', 'failed': '', 'running': '', 'queued': '⏳'}
        emoji = estado_emoji.get(run['state'], '')
        print(f"  {emoji} Run: {run['dag_run_id'][:40]:40s} | Estado: {run['state']:10s} | Fecha: {run['execution_date'][:19]}")
    print("-" * 80)
else:
    print(f" Error: {response.status_code}")

⏳ Esperando 10 segundos para que el DAG se ejecute...


 Ejecuciones del DAG 'dag_ejemplo' (55 total):
--------------------------------------------------------------------------------
   Run: manual__2026-03-21T15:15:06.756505+00:00 | Estado: success    | Fecha: 2026-03-21T15:15:06
   Run: manual__2026-03-21T15:15:22+00:00        | Estado: success    | Fecha: 2026-03-21T15:15:22
   Run: manual__2026-03-21T15:19:42.082134+00:00 | Estado: success    | Fecha: 2026-03-21T15:19:42
   Run: manual__2026-03-21T15:19:57+00:00        | Estado: success    | Fecha: 2026-03-21T15:19:57
   Run: manual__2026-03-21T15:21:05.589088+00:00 | Estado: success    | Fecha: 2026-03-21T15:21:05
   Run: manual__2026-03-21T15:21:20+00:00        | Estado: success    | Fecha: 2026-03-21T15:21:20
   Run: manual__2026-03-21T15:22:21.025974+00:00 | Estado: success    | Fecha: 2026-03-21T15:22:21
   Run: manual__2026-03-21T15:22:35+00:00        | Estado: success    | Fecha: 2026-03-21T15:22:35
   Run: manual__2026-03-21T15:28:50.807044+00:00 | Estado: success    | Fecha: 2

## 5. Consultar el estado de las tareas de una ejecución

Para una ejecución concreta, consulto el estado de cada tarea individual.

In [5]:
# Consultar las instancias de tareas de la última ejecución
response = session.get(
    f"{AIRFLOW_API_URL}/dags/{dag_id}/dagRuns/{dag_run_id}/taskInstances"
)

if response.status_code == 200:
    tasks = response.json()['task_instances']
    print(f" Tareas de la ejecución '{dag_run_id[:40]}':")
    print("-" * 70)
    for task in tasks:
        estado_emoji = {'success': '', 'failed': '', 'running': '', 'queued': '⏳', 'upstream_failed': '️'}
        emoji = estado_emoji.get(task['state'], '')
        duracion = task.get('duration', 'N/A')
        print(f"  {emoji} Tarea: {task['task_id']:25s} | Estado: {task['state']:15s} | Duración: {duracion}s")
    print("-" * 70)
else:
    print(f" Error: {response.status_code}")

 Tareas de la ejecución 'manual__2026-03-21T21:29:44.824365+00:00':
----------------------------------------------------------------------
   Tarea: tarea_saludo              | Estado: success         | Duración: 0.104794s
   Tarea: imprimir_fecha            | Estado: success         | Duración: 0.108765s
   Tarea: tarea_procesamiento       | Estado: success         | Duración: 2.119409s
----------------------------------------------------------------------


## 6. Consultar los logs de una tarea

Accedo al log de una tarea específica para ver su salida detallada.

In [6]:
# Consultar el log de la tarea 'tarea_saludo'
task_id = "tarea_saludo"
try_number = 1

response = session.get(
    f"{AIRFLOW_API_URL}/dags/{dag_id}/dagRuns/{dag_run_id}/taskInstances/{task_id}/logs/{try_number}",
    headers={'Accept': 'text/plain'}  # Los logs se devuelven en texto plano
)

if response.status_code == 200:
    print(f" Log de la tarea '{task_id}' (intento {try_number}):")
    print("=" * 70)
    # Mostramos las últimas 30 líneas del log para no saturar la salida
    log_lines = response.text.strip().split('\n')
    for line in log_lines[-30:]:
        print(line)
    print("=" * 70)
else:
    print(f" Error al obtener logs: {response.status_code}")
    print(response.text)

 Log de la tarea 'tarea_saludo' (intento 1):
6c0c1680f9ed
 INFO - ::group::Log message source details
*** Found local files:
***   * /opt/airflow/logs/dag_id=dag_ejemplo/run_id=manual__2026-03-21T21:29:44.824365+00:00/task_id=tarea_saludo/attempt=1.log
 INFO - ::endgroup::
[2026-03-21T21:29:46.610+0000] {local_task_job_runner.py:123} INFO - ::group::Pre task execution logs
[2026-03-21T21:29:46.614+0000] {taskinstance.py:2613} INFO - Dependencies all met for dep_context=non-requeueable deps ti=<TaskInstance: dag_ejemplo.tarea_saludo manual__2026-03-21T21:29:44.824365+00:00 [queued]>
[2026-03-21T21:29:46.617+0000] {taskinstance.py:2613} INFO - Dependencies all met for dep_context=requeueable deps ti=<TaskInstance: dag_ejemplo.tarea_saludo manual__2026-03-21T21:29:44.824365+00:00 [queued]>
[2026-03-21T21:29:46.618+0000] {taskinstance.py:2866} INFO - Starting attempt 1 of 2
[2026-03-21T21:29:46.629+0000] {taskinstance.py:2889} INFO - Executing <Task(PythonOperator): tarea_saludo> on 2026-0

## 7. Consultar XCom de una tarea

Los valores XCom son la forma en que las tareas de Airflow se comunican entre sí. Puedo consultar estos valores a través de la API REST.

In [7]:
# Consultar los XCom de la tarea 'tarea_saludo'
response = session.get(
    f"{AIRFLOW_API_URL}/dags/{dag_id}/dagRuns/{dag_run_id}/taskInstances/{task_id}/xcomEntries"
)

if response.status_code == 200:
    xcoms = response.json().get('xcom_entries', [])
    print(f" XCom entries de la tarea '{task_id}':")
    print("-" * 60)
    for xcom in xcoms:
        # Obtener el valor completo del XCom
        xcom_detail = session.get(
            f"{AIRFLOW_API_URL}/dags/{dag_id}/dagRuns/{dag_run_id}/taskInstances/{task_id}/xcomEntries/{xcom['key']}"
        )
        if xcom_detail.status_code == 200:
            valor = xcom_detail.json().get('value', 'N/A')
            print(f"   Key: {xcom['key']}")
            print(f"   Value: {valor}")
    print("-" * 60)
else:
    print(f" Error: {response.status_code}")

 XCom entries de la tarea 'tarea_saludo':
------------------------------------------------------------


   Key: return_value
   Value: {'status': 'ok', 'mensaje': '¡Hola desde el DAG de ejemplo!'}
------------------------------------------------------------


---

## 8. Uso de la CLI de Airflow desde Jupyter

Como Airflow está instalado en el mismo entorno (contenedor), puedo usar directamente la CLI con comandos`!airflow ...`.

In [8]:
# Listar todos los DAGs disponibles usando la CLI
!airflow dags list

/home/airflow/.local/lib/python3.10/site-packages/airflow/configuration.py:765 FutureWarning: The auth_backends setting in [api] has had airflow.api.auth.backend.session added in the running config, which is needed by the UI. Please update your config before Apache Airflow 3.0.


dag_id         | fileloc                                 | owners  | is_paused
===============+=========================================+=========+==========
dag_ejemplo    | /opt/airflow/dags/dag_ejemplo.py        | airflow | False    
dag_papermill  | /opt/airflow/dags/dag_papermill.py      | airflow | False    
workflow_video | /opt/airflow/dags/dag_workflow_video.py | airflow | False    
                                                                              


In [9]:
# Listar las tareas de un DAG específico
!airflow tasks list dag_ejemplo --tree

/home/airflow/.local/lib/python3.10/site-packages/airflow/configuration.py:765 FutureWarning: The auth_backends setting in [api] has had airflow.api.auth.backend.session added in the running config, which is needed by the UI. Please update your config before Apache Airflow 3.0.


<Task(PythonOperator): tarea_saludo>
    <Task(BashOperator): imprimir_fecha>
    <Task(PythonOperator): tarea_procesamiento>


In [10]:
# Lanzar un DAG desde la CLI
!airflow dags trigger dag_ejemplo

/home/airflow/.local/lib/python3.10/site-packages/airflow/configuration.py:765 FutureWarning: The auth_backends setting in [api] has had airflow.api.auth.backend.session added in the running config, which is needed by the UI. Please update your config before Apache Airflow 3.0.


[2026-03-21T21:29:59.974+0000] {__init__.py:43} INFO - Loaded API auth backend: airflow.api.auth.backend.basic_auth
[2026-03-21T21:29:59.974+0000] {__init__.py:43} INFO - Loaded API auth backend: airflow.api.auth.backend.session


     |      |      |      |      |      |      | last |     |      |     |      
     |      |      | data | data |      |      | _sch |     |      |     |      
     |      |      | _int | _int |      | exte | edul | log |      | sta |      
     |      | dag_ | erva | erva |      | rnal | ing_ | ica |      | rt_ |      
     | dag_ | run_ | l_st | l_en | end_ | _tri | deci | l_d | run_ | dat |      
conf | id   | id   | art  | d    | date | gger | sion | ate | type | e   | state
=====+======+======+======+======+======+======+======+=====+======+=====+======
{}   | dag_ | manu | 2026 | 2026 | None | True | None | 202 | manu | Non | queue
     | ejem | al__ | -03- | -03- |      |      |      | 6-0 | al   | e   | d    
     | plo  | 2026 | 21   | 21   |      |      |      | 3-2 |      |     |      
     |      | -03- | 21:3 | 21:3 |      |      |      | 1   |      |     |      
     |      | 21T2 | 0:00 | 0:00 |      |      |      | 21: |      |     |      
     |      | 1:30 | +00: | 

In [11]:
# Lanzar un DAG con parámetros JSON desde la CLI
!airflow dags trigger workflow_video --conf '{"descripcion": "Lanzado desde CLI en Jupyter", "commit": "0000"}'

/home/airflow/.local/lib/python3.10/site-packages/airflow/configuration.py:765 FutureWarning: The auth_backends setting in [api] has had airflow.api.auth.backend.session added in the running config, which is needed by the UI. Please update your config before Apache Airflow 3.0.


[2026-03-21T21:30:01.870+0000] {__init__.py:43} INFO - Loaded API auth backend: airflow.api.auth.backend.basic_auth
[2026-03-21T21:30:01.870+0000] {__init__.py:43} INFO - Loaded API auth backend: airflow.api.auth.backend.session


      |      |      |      |      |      | ext | last |     |      |     |      
      |      |      | data | data |      | ern | _sch |     |      |     |      
      |      |      | _int | _int |      | al_ | edul | log |      | sta |      
      |      | dag_ | erva | erva |      | tri | ing_ | ica |      | rt_ |      
      | dag_ | run_ | l_st | l_en | end_ | gge | deci | l_d | run_ | dat |      
conf  | id   | id   | art  | d    | date | r   | sion | ate | type | e   | state
======+======+======+======+======+======+=====+======+=====+======+=====+======
{'des | work | manu | 2026 | 2026 | None | Tru | None | 202 | manu | Non | queue
cripc | flow | al__ | -03- | -03- |      | e   |      | 6-0 | al   | e   | d    
ion': | _vid | 2026 | 20   | 21   |      |     |      | 3-2 |      |     |      
      | eo   | -03- | 21:3 | 21:3 |      |     |      | 1   |      |     |      
'Lanz |      | 21T2 | 0:02 | 0:02 |      |     |      | 21: |      |     |      
ado   |      | 1:30 | +00: |

In [12]:
# Ver la información general de Airflow
!airflow info

/home/airflow/.local/lib/python3.10/site-packages/airflow/configuration.py:765 FutureWarning: The auth_backends setting in [api] has had airflow.api.auth.backend.session added in the running config, which is needed by the UI. Please update your config before Apache Airflow 3.0.



Apache Airflow
version                | 2.10.4                                             
executor               | SequentialExecutor                                 
task_logging_handler   | airflow.utils.log.file_task_handler.FileTaskHandler
sql_alchemy_conn       | sqlite:////opt/airflow/db/airflow.db               
dags_folder            | /opt/airflow/dags                                  
plugins_folder         | /opt/airflow/plugins                               
base_log_folder        | /opt/airflow/logs                                  
remote_base_log_folder |                                                    
                                                                            

System info
OS              | Linux                                                         
architecture    | x86_64                                                        
uname           | uname_result(system='Linux', node='8f5ccd2c4409',             
                | release='6.6.87.2

In [13]:
# Ver el estado de las últimas ejecuciones de un DAG
!airflow dags list-runs -d dag_ejemplo --limit 5

/home/airflow/.local/lib/python3.10/site-packages/airflow/configuration.py:765 FutureWarning: The auth_backends setting in [api] has had airflow.api.auth.backend.session added in the running config, which is needed by the UI. Please update your config before Apache Airflow 3.0.


Usage: airflow [-h] GROUP_OR_COMMAND ...

Positional Arguments:
  GROUP_OR_COMMAND

    Groups
      config         View configuration
      connections    Manage connections
      dags           Manage DAGs
      db             Database operations
      jobs           Manage jobs
      pools          Manage pools
      providers      Display providers
      roles          Manage roles
      tasks          Manage tasks
      users          Manage users
      variables      Manage variables

    Commands:
      cheat-sheet    Display cheat sheet
      dag-processor  Start a standalone Dag Processor instance
      info           Show information about current Airflow and environment
      kerberos       Start a kerberos ticket renewer
      plugins        Dump information about loaded plugins
      rotate-fernet-key
                     Rotate encrypted connection credentials and variables
      scheduler      Start a scheduler instance
      standalone     Run an all-in-one copy of Airf

---

## 9. Ventajas de usar Jupyter como cliente de Airflow

###  Para pruebas (Testing)
- Permite **probar la API REST de Airflow de forma interactiva**, viendo los resultados inmediatamente.
- Se pueden **lanzar DAGs con diferentes configuraciones** y observar los resultados sin necesidad de editar archivos de configuración.
- Facilita la **depuración** al poder inspeccionar logs, estados y XCom de forma programática.

###  Para automatización
- Se pueden crear **scripts reutilizables** en notebooks que automaticen operaciones comunes sobre Airflow.
- Permite **orquestar múltiples DAGs** desde un único notebook, útil para pipelines de datos complejos.
- La combinación de API REST + CLI permite **cubrir cualquier caso de uso** de automatización.

###  Para aprendizaje
- El formato de notebook permite **documentar y ejecutar** al mismo tiempo, creando tutoriales interactivos.
- Se puede **experimentar** con la API sin riesgo, cambiando parámetros y observando el comportamiento.
- Las celdas Markdown permiten **añadir explicaciones** junto al código, facilitando la comprensión.

###  Ventaja adicional: Visualización
- Desde Jupyter se pueden **visualizar métricas y resultados** de las ejecuciones de Airflow usando librerías como matplotlib o pandas.
- Permite crear **dashboards ad-hoc** para monitorizar el estado de los workflows.